setting up my Azure client

In [2]:
%pip install setuptools==81.0.0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import Data
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import AccountKeyConfiguration
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

Found the config file in: /config.json


get the datastore

In [7]:
# get the datastore
datastore = ml_client.datastores.get_default()

# build the path to the raw file
datastore_path = f"azureml://datastores/{datastore.name}/paths/Online Retail.xlsx"

# register it as a data asset
retail_data = Data(
    path=datastore_path,
    name="online-retail-raw",
    version="1",
    type=AssetTypes.URI_FILE,
    description="Raw UCI Online Retail transaction dataset for customer segmentation"
)

registered_data = ml_client.data.create_or_update(retail_data)

In [8]:
# confirm set up
print("Name:", registered_data.name)
print("Version:", registered_data.version)
print("Type:", registered_data.type)
print("Path:", registered_data.path)

Name: online-retail-raw
Version: 1
Type: uri_file
Path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/clusterdatastore/paths/Online Retail.xlsx


conversion of excel to parquet

In [9]:
# retrieving the raw data

raw_data = ml_client.data.get(
    name="online-retail-raw",
    version="1"
)

print(raw_data.path)

azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/clusterdatastore/paths/Online Retail.xlsx


package for easy access to azure ml storage URIs by Python/Panda

In [10]:
%pip install -U azureml-fsspec openpyxl pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 67.5 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 94.1 MB/s  0:00:006m0:00:0100:01
  Attempting uninstall: azureml-dataprep-rslex
    Found existing installation: azureml-dataprep-rslex 2.22.5
    Uninstalling azureml-dataprep-rslex-2.22.5:
      Successfully uninstalled azureml-dataprep-rslex-2.22.5
  Attempting uninstall: pyarrow━━━━━━━━━━━━━━━━━ 0/8 [azureml-dataprep-rslex]
    Found existing installation: pyarrow 24.0.0m 0/8 [azureml-dataprep-rslex]
    Uninstalling pyarrow-24.0.0:━━━━━━━━━━━━ 0/8 [azureml-dataprep-rslex]
      Successfully uninstalled pyarrow-24.0.0 0/8 [azureml-dataprep-rslex]
  Attempting uninstall: azureml-dataprep-native━━━━━━━━━━━━━━━━━━━ 1/8 [pyarrow]
    Found existing installation: azureml-dataprep-native 41.0.0 1/8 [pyarrow]
    Uninstalling azureml-dataprep-native-41.0.0:━━━━━━━━━━━━━━ 1/8 [pyarrow]
      Successfully uninstalled azureml-dataprep-native-41.0.0

In [13]:
#read the registered excel asset

import pandas as pd

df = pd.read_excel(
    raw_data.path,
    engine="openpyxl"
)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attem

In [14]:
df.shape

(541909, 8)

In [22]:
# conversion for parquet
text_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Country"
]

for column in text_columns:
    df[column] = df[column].astype("string")

In [23]:
df.dtypes

InvoiceNo              string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                string
dtype: object

In [24]:
# converting to parquet locally
parquet_path = "../data/processed/online_retail.parquet"

df.to_parquet(
    parquet_path,
    engine="pyarrow",
    index=False
)

In [25]:
# verifying the parquet before loading
df_parquet = pd.read_parquet(parquet_path)

print(df_parquet.shape)
df_parquet.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [26]:
# confirming if my data structure is still retained
print("Excel shape:", df.shape)
print("Parquet shape:", df_parquet.shape)

Excel shape: (541909, 8)
Parquet shape: (541909, 8)


In [27]:
# final schema check
print(df_parquet.dtypes)

InvoiceNo              string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                string
dtype: object


uploading the local parquet to datastore

In [6]:
parquet_asset = Data(
    name="online-retail-parquet",
    version="2",
    type=AssetTypes.URI_FILE,
    path="../data/processed/online_retail.parquet"
)

In [7]:
# register as a data asset

registered_parquet = ml_client.data.create_or_update(parquet_asset)

In [8]:
# confirming everything set up prope
print("Name:", registered_parquet.name)
print("Version:", registered_parquet.version)
print("Type:", registered_parquet.type)
print("Path:", registered_parquet.path)

Name: online-retail-parquet
Version: 2
Type: uri_file
Path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/fa3d1d5709e7c559fc2f24c7bc3d851da344356801fd417a88501eb84a703592/online_retail.parquet


In [9]:
# getting path
parquet_asset = ml_client.data.get(
    name="online-retail-parquet",
    version="2"
)

print(parquet_asset.path)

azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/fa3d1d5709e7c559fc2f24c7bc3d851da344356801fd417a88501eb84a703592/online_retail.parquet


register the data asset as mltable

In [10]:
# creating an mltable object in memory
parquet_uri = registered_parquet.path

tbl = mltable.from_parquet_files(
    paths=[{
        "file":parquet_uri
    }]
)

In [11]:
# confirm structure
test_df = tbl.to_pandas_dataframe()

print(test_df.shape)

test_df.head()

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [12]:
# a folder for the mltable definition
import os 

mltable_folder = "../data/mltable/online-retail"

os.makedirs(
    mltable_folder,
    exist_ok=True
)

# save the mltable definition
tbl.save(mltable_folder)

paths:
- file: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/fa3d1d5709e7c559fc2f24c7bc3d851da344356801fd417a88501eb84a703592/online_retail.parquet
transformations:
- read_parquet:
    include_path_column: false
    path_column: Path
type: mltable

In [13]:
# registering the mltable as azure asset
mltable_asset = Data(
    name="online-retail-mltable",
    path=mltable_folder,
    version="1",
    type=AssetTypes.MLTABLE,
    description="MLTable for the UCI Online Retail customer segmentation dataset"
)  

registered_mltable = ml_client.data.create_or_update(
    mltable_asset
)

In [16]:
# verifying setup
print("Name:", registered_mltable.name)
print("Version:", registered_mltable.version)
print("Type:", registered_mltable.type)
print("Path:", registered_mltable.path)

Name: online-retail-mltable
Version: 1
Type: mltable
Path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/b342c082372b11e39f0569465f71c3e153a7f63f11582709b2362518245ca18b/online-retail/
